# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Paggawa ng iyong unang aplikasyon para sa pagbuo ng imahe

Ano ba ang kailangan upang makagawa ng isang aplikasyon para sa pagbuo ng imahe? Kailangan mo ang mga sumusunod na library:

- **python-dotenv**, lubos na inirerekomenda gamitin mo ang library na ito upang itago ang iyong mga sikreto sa isang *.env* na file na hiwalay sa code.
- **openai**, ang library na ito ang gagamitin mo para makipag-ugnayan sa OpenAI API.
- **pillow**, para sa pagtrabaho sa mga imahe sa Python.


1. Gumawa ng isang *.env* na file na may sumusunod na nilalaman:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kolektahin ang mga nabanggit na librarya sa isang file na tinatawag na *requirements.txt* gaya nito:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Susunod, gumawa ng virtual environment at i-install ang mga librarya:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para sa Windows, gamitin ang mga sumusunod na utos upang gumawa at i-activate ang iyong virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Idagdag ang sumusunod na code sa file na tinatawag na *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Ipaliwanag natin ang code na ito:

- Una, ini-import natin ang mga library na kailangan natin, kasama ang OpenAI library, ang dotenv library, ang base64 module, at ang Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Pagkatapos nito, ginagawa natin ang client, na nagbabasa ng API key mula sa iyong ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Sunod, gumagawa tayo ng imahe:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models ay nagbabalik ng imahe bilang isang **base64** string sa `data[0].b64_json`. Kino-convert natin ito pabalik sa byte at isinusulat sa file — walang URL para i-download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Sa huli, binubuksan natin ang imahe at ginagamit ang karaniwang image viewer para ipakita ito:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mas detalyadong impormasyon sa paggawa ng imahe

Tingnan natin ang mga parameter ng `images.generate`:

- **model**, ay ang image model na gagamitin (halimbawa `gpt-image-2`).
- **prompt**, ay ang tekstong prompt na ginagamit para gumawa ng imahe. Dito ay "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, ay ang sukat ng ginawa na imahe (halimbawa `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, ay ang bilang ng mga larawan na ginawa. Dito ay gumawa tayo ng isa.

> Ang mga image model ay hindi tumatanggap ng parameter na `temperature` — iyon ay kontrol para sa pag-generate ng teksto. Para magkaroon ng variety, tawagin muli ang API; para mabawasan ang variety, gawing mas tiyak ang iyong prompt.

## Karagdagang kakayahan ng paggawa ng imahe

Nakita mo kung paano gumawa ng imahe gamit ang iilang linya ng Python. Ang `gpt-image` models ay maaari ding **mag-edit** ng umiiral na imahe. Sa pamamagitan ng pagbibigay ng imahe, opsyonal na **mask** (na nagtatalaga sa bahagi na babaguhin), at isang prompt, maari mong baguhin ang isang bahagi ng imahe — halimbawa, magdagdag ng sumbrero sa ating kuneho.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Ang base image ay naglalaman lamang ng kuneho; ang huling imahe ay may sumbrero sa kuneho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
